# Big Sur Marathon

The course across stacked elevation bands from USGS, with Heartbreak Hill drawn to the same scale. Run `fetch_week1.py day2` first.

## 1 · Settings

In [ ]:
DAY      = 2
PROMPT   = "Lines"
SLUG     = "lines-big-sur"
NOTEBOOK = "day02_lines_big_sur.ipynb"
TITLE    = "Big Sur\nMarathon"
CREDITS  = "Course facts: Big Sur International Marathon. Route: © OpenStreetMap contributors. Elevation and contours: USGS 3DEP."

## 2 · Load

In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy import ndimage
from mapkit import poster, publish
from mapkit.paths import PROCESSED

route = pd.read_csv(PROCESSED / "bigsur_profile.csv")
boston = pd.read_csv(PROCESSED / "boston_comm_ave.csv")
dem_file = np.load(PROCESSED / "bigsur_dem.npz")
dem = dem_file["dem_dm"] / 10.0
x0, y0, x1, y1 = dem_file["bbox_3857"]

## 3 · Analysis, every number on the map comes from here

In [ ]:
def flatten_bridges(df):
    """Elevation models read the canyon under a bridge, so we draw a straight deck across each one."""
    z = df.elev_ft.copy()
    z[df.bridge] = np.nan
    return z.interpolate(limit_direction="both")

route["z"] = flatten_bridges(route)
boston["z"] = boston.elev_ft.rolling(5, center=True, min_periods=1).mean()   # smooth out street level noise

def biggest_climb(df):
    """The largest rise from any low point to a later high point."""
    low = df.z.cummin()
    top = (df.z - low).idxmax()
    bottom = df.z[:top + 1].idxmin()
    return bottom, top, float(df.z[top] - df.z[bottom])

# Hurricane Point: from the Little Sur River to the top of the course
a = route[(route.dist_mi >= 9.5) & (route.dist_mi <= 12.5)]
hp_low, hp_top, hp_rise = biggest_climb(a)
hp_miles = route.dist_mi[hp_top] - route.dist_mi[hp_low]
# Heartbreak Hill: the last big climb on Commonwealth Avenue
b = boston[(boston.dist_mi >= 1.0) & (boston.dist_mi <= 2.0)]    # past the earlier Newton hills
h_low, h_top, h_rise = biggest_climb(b)
h_miles = boston.dist_mi[h_top] - boston.dist_mi[h_low]
ratio = hp_rise / h_rise
print(f"Route length       {route.dist_mi.max():.2f} mi  (official 26.2)")
print(f"Top of the course  {route.z.max():.0f} ft  (official 560)")
print(f"Hurricane Point    {hp_rise:.0f} ft over {hp_miles:.2f} mi, mile {route.dist_mi[hp_low]:.1f} to {route.dist_mi[hp_top]:.1f}  (official chart: 40 to 560 ft = 520)")
print(f"Heartbreak Hill    {h_rise:.0f} ft over {h_miles:.2f} mi  (published 88 to 91)")
print(f"Ratio              {ratio:.1f}x")
print(f"Total climbing     {route.z.diff().clip(lower=0).sum():.0f} ft  (published 2,182)")

## 4 · Map

In [ ]:
NIGHT, BASE, GOLD = "#1b2a4e", "#141f3c", "#ffd166"
fig, ax = poster.page(NIGHT)
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.15, 36.105, -121.72, 36.565)

# clean up bad pixels, and keep only the mainland
bad = dem > 1800
if bad.any():
    dem[bad] = ndimage.median_filter(dem, size=9)[bad]
labels, _ = ndimage.label(ndimage.binary_opening(dem > 1, iterations=2))
sizes = np.bincount(labels.ravel())
sizes[0] = 0
is_land = labels == sizes.argmax()
smooth = ndimage.gaussian_filter(np.where(is_land, dem, 0), 1.8)
gx = np.linspace(x0, x1, dem.shape[1])
gy = np.linspace(y1, y0, dem.shape[0])

# the ocean: lines of swell rolling in, fading as they leave the shore
to_shore = ndimage.distance_transform_edt(~is_land) * (x1 - x0) / dem.shape[1]
for i, d in enumerate(np.arange(350, 7000, 350)):
    ax.contour(gx, gy, to_shore, levels=[d], colors="#9fc5e8", linewidths=0.7, alpha=max(0.05, 0.62 - i * 0.032), zorder=1)

# the land: stacked bands of color, sand at the shore to deep plum at the summits
sunset = LinearSegmentedColormap.from_list("sunset", ["#fbe3b5", "#f9b97a", "#f2805c", "#d94f6a", "#9c3381", "#4f2277"])
levels = [1] + list(range(100, 1500, 100)) + [1800]
ax.contourf(gx, gy, np.where(is_land, smooth, np.nan), levels=levels, cmap=sunset, zorder=2)
ax.contour(gx, gy, np.where(is_land, smooth, np.nan), levels=levels[1:-1], colors="#3b1a4f", linewidths=0.45, alpha=0.38, zorder=3)
ax.contour(gx, gy, is_land.astype(float), levels=[0.5], colors="#fff4dc", linewidths=1.3, zorder=3)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# the course: a dark thread with a white edge, and the big climb in gold
rx, ry = poster.merc(route.lon, route.lat)
ax.plot(rx, ry, color="white", lw=6, solid_capstyle="round", zorder=5)
ax.plot(rx, ry, color="#20163a", lw=2.8, solid_capstyle="round", zorder=6)
climb = route.loc[hp_low:hp_top]
cx, cy = poster.merc(climb.lon, climb.lat)
ax.plot(cx, cy, color="#20163a", lw=8.5, solid_capstyle="round", zorder=7)
ax.plot(cx, cy, color=GOLD, lw=5, solid_capstyle="round", zorder=8)

def mark(mile_at, text, side="right", bold=False):
    i = (route.dist_mi - mile_at).abs().idxmin()
    mx, my = poster.merc(route.lon[i], route.lat[i])
    ax.scatter([mx], [my], s=46, color="white", edgecolor="#20163a", lw=1.4, zorder=9)
    gap = (fx1 - fx0) * 0.016
    poster.halo_text(ax, mx + (gap if side == "right" else -gap), my, text, color="#20163a" if side == "right" else "white", size=13.5,
                     path=poster.BARLOW_BOLD if bold else poster.BARLOW, halo="#fff0d6" if side == "right" else NIGHT, halo_width=3.4, va="center", zorder=10,
                     ha="left" if side == "right" else "right")

mark(0, "Start: Big Sur", bold=True)
mark(11, "Hurricane Point climb, miles 10 to 12", bold=True)
mark(13.2, "Bixby Bridge, a grand piano plays here")
mark(route.dist_mi.max(), "Finish: Carmel", side="left", bold=True)

# a dark base holds the elevation profile, with Heartbreak Hill to the same scale
ax.add_patch(plt.Rectangle((fx0, fy0), fx1 - fx0, (fy1 - fy0) * 0.205, color=BASE, zorder=11))
strip = fig.add_axes([0.06, 0.082, 0.88, 0.095], facecolor="none")
strip.fill_between(route.dist_mi, route.z, color="#6a7ba6", alpha=0.5, lw=0)
strip.plot(route.dist_mi, route.z, color="white", lw=1.2)
strip.fill_between(climb.dist_mi, climb.z, color=GOLD, lw=0)
hb = boston.loc[h_low:h_top]
hb_start = 28.0
hb_x = hb_start + hb.dist_mi - hb.dist_mi.iloc[0]
strip.fill_between(hb_x, hb.z - boston.z[h_low], color=GOLD, lw=0)
strip.set_xlim(0, 33.5); strip.set_ylim(0, 640)
strip.axhline(0, color="#8ea0c4", lw=0.7)
for s in strip.spines.values():
    s.set_visible(False)
strip.set_xticks([0, 5, 10, 15, 20, 26.2], ["0", "5", "10", "15", "20", "26.2 miles"], fontproperties=poster.font(poster.BARLOW, 11), color="#c5d0e6")
strip.tick_params(axis="x", length=0, pad=3)
strip.set_yticks([])
strip.text(route.dist_mi[hp_top] + 0.35, 590, f"Hurricane Point: {hp_rise:.0f} ft", color=GOLD, fontproperties=poster.font(poster.BARLOW_BOLD, 13), va="center")
strip.text(hb_start - 0.1, 165, f"Boston's Heartbreak Hill\n{h_rise:.0f} ft, same scale", color=GOLD, fontproperties=poster.font(poster.BARLOW_BOLD, 12), va="bottom", linespacing=1.25)

poster.title(fig, TITLE, size=64, x=0.06, y=0.962, color="#fff4dc", path=poster.DM_SERIF, sub_path=poster.BARLOW, line=1.03,
             subtitle="The world's largest rural marathon: 26.2 miles\non California's Highway 1, with one climb over\nfive times the size of Heartbreak Hill.", sub_size=16, sub_color="#dfe7f5")
poster.credit(fig, CREDITS, color="#9eabc7", path=poster.BARLOW, name_path=poster.BARLOW_BOLD, name_color="#fff4dc")

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"The course on contour lines, and a climb over five times Heartbreak Hill", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)